# Working with LLMs — Conversations

### Chat Completions and the Responses API

**From zero to a working, memory-enabled chatbot — and then the API that replaces it.**

In the first half we build a chatbot on the **Chat Completions API**. By the end of that half it will:

- Maintain a **persistent persona** via system prompts
- **Remember** the full conversation history
- **Stream** responses token-by-token for a real-time feel
- **Handle errors** gracefully so it never crashes
- **Track token usage** so you always know what a call costs

In the second half we move to the **Responses API** — OpenAI's newer and recommended surface — and
spend real time on the two controls people most often get wrong: `instructions` and the `developer` role.

> **Prerequisites:** A valid OpenAI API key. This notebook reads it from an `.env` file
> containing one line: `OPENAI_API_KEY=sk-...`

### Where we are going

```
 Part 1   Setup ....................  keys, client, what models you can call
 Part 2   Chat Completions .........  roles, persona, memory
 Part 3   Parameters ...............  temperature, max_tokens, the system prompt
 Part 4   Streaming ................  token-by-token, the "typing" effect
 Part 5   Production hygiene .......  errors, tokens, cost
 ─────────────────────────────────────────────────────────────────────────
 Part 6   Responses API ............  structure, parameters, server-side threads
 Part 7   instructions vs developer   who sets the rules, and for how long
```

Parts 1-5 are one API. Parts 6-7 are the other one. The line in the middle is the moment
you stop hand-carrying conversation history and let the server do it.

## Part 1 — Setup

**Goal:** install the SDK, load credentials safely, and see what the key can actually call.

---
### 1.1 Install dependencies

In [2]:
# Run once to install required packages
#!pip install openai python-dotenv tiktoken ipython-autotime -q

In [3]:
# Prints how long every cell took, right under the cell. Handy in a live class.
%load_ext autotime

time: 195 µs (started: 2026-08-25 10:08:14 +05:30)


### 1.2 Load your API key securely

We **never** hardcode API keys. They live in a `.env` file that we load at runtime with
`python-dotenv`. Treat the key like a password — if it leaks, anyone can spend your money.

In [4]:
import os
import textwrap

from dotenv import load_dotenv


def pretty_print(*args):
    """print(), but wrapped at 80 columns so long model answers stay readable."""
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception:
        print(text)  # fallback if the text is not wrappable


load_dotenv('/Users/shivam13juna/Documents/scaler/GEN_AI_REF/openai_key.env')
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.
time: 5.74 ms (started: 2026-08-25 10:08:14 +05:30)


In [5]:
import truststore
truststore.inject_into_ssl()

# Optional. Needed on machines behind a VPN / corporate proxy that rewrites certificates.

time: 29.5 ms (started: 2026-08-25 10:08:14 +05:30)


### 1.3 Initialize the OpenAI client

The `OpenAI` class is our gateway to every model endpoint. Create it once, reuse it everywhere.

In [6]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

OpenAI client ready.
time: 694 ms (started: 2026-08-25 10:08:14 +05:30)


In [7]:
models = client.models.list()

# These are the models this API key has access to.
for m in models.data:
    pretty_print(m.id)

gpt-4-0613
gpt-4
gpt-3.5-turbo
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
omni-moderation-latest
omni-moderation-2024-09-26
o1-2024-12-17
o1
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-search-preview
gpt-4o-transcribe
gpt-4o-mini-transcribe
o1-pro-2025-03-19
o1-pro
gpt-4o-mini-tts
o3-2025-04-16
o4-mini-2025-04-16
o3
o4-mini
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2025-04-14
gpt-4.1-mini
gpt-4.1-nano-2025-04-14
gpt-4.1-nano
gpt-image-1
gpt-4o-transcribe-diarize
gpt-5-chat-latest
gpt-5-2025-08-07
gpt-5
gpt-5-mini-2025-08-07
gpt-5-mini
gpt-5-nano-2025-08-07
gpt-5-nano
gpt-audio-2025-08-28
gpt-realtime
gpt-realtime-2025-08-28
gpt-audio
gpt-5-codex
gpt-image-1-mini
gpt-5-pro-2025-10

## Part 2 — Chat Completions: roles, persona and memory

Three ideas power every conversational system built on the Chat Completions API:

| Concept | How it works |
|---|---|
| **System prompt** | A special message at the top of the conversation that tells the model *who it is* and *how to behave*. It persists across every turn. |
| **Message history** | We keep a running Python list of every user and assistant message. The model sees the full list on every call — that is how it "remembers" earlier turns. |
| **Roles** | Every message carries a role — `system`, `user`, or `assistant` — so the model knows who said what. |

The single most important thing to internalise: **the API is stateless.** The model remembers
nothing between calls. "Memory" is just you resending the whole list every time.

In [8]:
current_messages = [
    {"role": "system", "content": "You are a little bit sarcastic and unhelpful, but in the end you answer the question."},
    {"role": "user", "content": "What is the capital of France?"}
]

response = client.chat.completions.create(
    model="gpt-4o",
    messages=current_messages
)

reply = response.choices[0].message.content
pretty_print(reply)

Oh, that's a real stumper! It's not like it's common knowledge or anything...
*drumroll please*... The capital of France is Paris. Enjoy your trip to the city
of lights!
time: 1.4 s (started: 2026-08-25 10:08:17 +05:30)


Now the memory part. We append the assistant's own reply back into the list, then ask a
follow-up that is only answerable if the earlier turns are present.

In [9]:
current_messages.append({"role": "assistant", "content": reply})
current_messages.append({"role": "user", "content": "What did I ask previously?"})

response = client.chat.completions.create(
    model="gpt-4o",
    messages=current_messages
)

reply = response.choices[0].message.content
pretty_print(reply)

Oh, playing the memory game, are we? Well, you had the audacity to ask the mind-
boggling question about the capital of France. In case you've already forgotten,
it's Paris.
time: 1.3 s (started: 2026-08-25 10:08:18 +05:30)


### 2.1 The same idea, as a real chat loop

Run it, chat for a few turns, then ask something like *"What was my first question?"* to see
the memory working. Type `quit` to stop.

In [10]:
# --- Persona & Memory Chat ---

messages = [
    {
        "role": "system",
        "content": (
            "You are a polite and clear Python tutor. "
            "You explain concepts with simple analogies and short code examples. "
            "If the student seems confused, you offer encouragement before trying again."
        )
    }
]

pretty_print("Python Tutor Bot (type 'quit' to exit)")
pretty_print("-" * 45)

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        pretty_print("Session ended.")
        break

    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=messages
    )

    reply = response.choices[0].message.content
    pretty_print(f"Assistant: {reply}\n")

    # Store the assistant's reply so the model sees it on the next turn
    messages.append({"role": "assistant", "content": reply})

Python Tutor Bot (type 'quit' to exit)
---------------------------------------------
You: What is a list comprehension?


Assistant: A list comprehension is a concise way to create a new list by
applying a rule to each item in an existing iterable (like a list), and
optionally filtering some items out.  Syntax (basic): - new_list = [expression
for item in iterable if condition]  - expression: what to put in the new list
(can use item) - item: each element from the iterable - iterable: something you
can loop over (list, range, etc.) - if condition: optional; keeps items where
condition is True  Examples: - Squares of numbers 0 through 9   squares = [x*x
for x in range(10)]  - Even numbers from 0 to 19   evens = [n for n in range(20)
if n % 2 == 0]  - Uppercase words   words = [w.upper() for w in ["cat", "dog",
"bird"]]  - Flatten a list of lists   nested = [[1, 2], [3, 4], [5]]   flat =
[item for sublist in nested for item in sublist]  Notes: - If you don’t want a
list right away, you can make a generator instead (lazy evaluation) using
parentheses:   squares_gen = (x*x for x in range(10))   # and later:
l

Assistant: Your first question was: "What is a list comprehension?" It was about
Python list comprehensions. Want to dive into one now?
You: quit
Session ended.
time: 10.9 s (started: 2026-08-25 10:08:19 +05:30)


### Part 2 — Think about it

1. **Why does the system message sit at the top of the list?**
   Because the model reads messages in order. Placing it first means every subsequent turn is
   interpreted through that persona lens.

2. **What happens if we only send the latest user message?**
   The model loses all context — it cannot reference earlier questions, correct itself, or hold a
   coherent thread. Each call becomes a fresh, one-shot interaction.

---

## Part 3 — Models and parameters

### 3.1 The model menu — choosing the right tool

Not every task needs the most expensive model.

| Model | Strengths | Best for |
|---|---|---|
| **gpt-4o-mini** | Fast, cheap, supports `temperature` | High-volume bots, drafts, anything needing reproducibility |
| **gpt-4o** | Multimodal (text + image + audio), fast | Vision tasks, long documents, versatile apps |
| **gpt-5-nano / gpt-5-mini** | Reasoning models, cheap for their class | Cost-sensitive production apps that need real thinking |
| **gpt-5** | Deepest reasoning | Math, science, hard multi-step problems |

> **Honest caveats:** all of these can hallucinate — confidently produce wrong answers — and
> most of them are verbose by default. Always verify critical outputs.

### 3.2 Parameters that shape output

Two parameters you will reach for constantly:

| Parameter | What it does | Typical values |
|---|---|---|
| `temperature` | Controls randomness. **0** = deterministic (same input → same output). **1** = creative and varied. | 0 for factual tasks, 0.7-0.9 for creative work |
| `max_tokens` | Hard cap on how many tokens the model may generate in its reply. | 256 for short answers, 1024+ for essays |

> **Important — and this catches people out:** the demos below use `gpt-4o-mini`, **not** `gpt-5-nano`.
> The GPT-5 family are *reasoning* models and they reject **both** of these parameters:
> `temperature` must be left at its default, and `max_tokens` has been replaced by
> `max_completion_tokens`. We come back to this properly in Part 6.

In [11]:
# --- Experiment: temperature ---

prompt_messages = [
    {"role": "system", "content": "You are a creative storyteller."},
    {"role": "user", "content": "Describe a sunset in one sentence."}
]

pretty_print("temperature=0  (deterministic)")
for i in range(3):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=prompt_messages,
        temperature=0,
        max_tokens=60
    )
    pretty_print(f"   Run {i+1}: {r.choices[0].message.content}")

print()
pretty_print("temperature=0.9  (creative)")
for i in range(3):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=prompt_messages,
        temperature=0.9,
        max_tokens=60
    )
    pretty_print(f"   Run {i+1}: {r.choices[0].message.content}")

temperature=0  (deterministic)


   Run 1: The sun dipped below the horizon, painting the sky in a breathtaking
tapestry of fiery oranges and soft purples, as the day whispered its final
goodbyes to the world.


   Run 2: The sun dipped below the horizon, painting the sky in a breathtaking
tapestry of fiery oranges and soft purples, as the day whispered its final
goodbyes to the world.


   Run 3: The sun dipped below the horizon, painting the sky in a breathtaking
tapestry of fiery oranges and soft purples, as the day whispered its final
goodbyes to the world.

temperature=0.9  (creative)


   Run 1: The horizon blazed with a tapestry of fiery oranges and soft purples,
as the sun dipped below the edge of the world, casting a tranquil glow over the
tranquil sea.


   Run 2: The sun dipped beneath the horizon, painting the sky in a breathtaking
tapestry of fiery oranges, soft pinks, and deep purples, while the day sighed
into a tranquil embrace of twilight.


   Run 3: The sun descended in a spectacular blaze of crimson and gold, casting
a warm glow over the horizon as the sky transformed into a canvas of swirling
pastels, bidding farewell to the day with a tranquil sigh.
time: 5.56 s (started: 2026-08-25 10:08:30 +05:30)


Notice how `temperature=0` produces near-identical output each run, while `temperature=0.9`
gives you variety. This is the main dial for creativity vs. consistency.

In [12]:
# --- Experiment: max_tokens ---

r_short = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain binary search."}],
    max_tokens=30
)

r_long = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain binary search."}],
    max_tokens=300
)

pretty_print("max_tokens=30:")
pretty_print(r_short.choices[0].message.content)
print()
pretty_print("max_tokens=300:")
pretty_print(r_long.choices[0].message.content)

max_tokens=30:
Binary search is an efficient algorithm for finding a target value in a sorted
array or list. It works on the principle of divide and conquer by repeatedly
dividing

max_tokens=300:
Binary search is an efficient algorithm used to find an item in a sorted list or
array. It operates by repeatedly dividing the search interval in half and
eliminating half of the remaining elements. The process consists of the
following steps:  1. **Initial Setup**: Start with two pointers: one pointing to
the beginning of the list (`low`) and the other pointing to the end (`high`).
2. **Calculate Midpoint**: Calculate the middle index of the current search
interval using the formula:    \[    \text{mid} = \text{low} +
\frac{(\text{high} - \text{low})}{2}    \]    (In some programming languages,
it's common to use integer division.)  3. **Comparison**:    - If the target
value is equal to the element at the `mid` index, the search is successful, and
the index of the `mid` element is returned.

`max_tokens` is a **guillotine, not a summariser**. The short answer above does not end early
because the model wrapped up — it ends mid-thought because it ran out of budget. If you want a
short answer, ask for one in the prompt; use `max_tokens` as a cost ceiling, not a style control.

### 3.3 Switching the system prompt = switching the personality

The system prompt is the single most powerful lever you have. Same model, same question,
completely different product.

In [13]:
funny_messages = [
    {
        "role": "system",
        "content": (
            "You are a stand-up comedian who explains programming concepts "
            "using jokes and funny analogies. Keep answers short and punchy."
        )
    },
    {"role": "user", "content": "What is recursion?"}
]

r = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=funny_messages,
    temperature=0.8
)

pretty_print(r.choices[0].message.content)

Recursion is like asking a mirror to tell you a joke about itself. It keeps
asking for a reflection until it finally runs out of mirrors and just laughs at
its own image! Just remember, if the mirror starts asking for a friend, you
might be in trouble!
time: 1.16 s (started: 2026-08-25 10:08:40 +05:30)


### Part 3 — Think about it

1. **Which model for a chatbot that must understand uploaded images?** → `gpt-4o` (multimodal).
2. **Tight budget, simple Q&A?** → `gpt-4o-mini`.
3. **Need the exact same answer every single run?** → a non-reasoning model with `temperature=0`.
   You cannot get that from the GPT-5 family.

---

## Part 4 — Streaming responses

### 4.1 Why stream?

When a model generates 500 tokens, the non-streaming approach makes you wait for *all 500*
before showing anything. Streaming sends tokens as they are produced, so the user sees the first
word in milliseconds. This is exactly how ChatGPT's "typing" effect works.

| Approach | User sees first word after... | Total time |
|---|---|---|
| Non-streaming | the full generation finishes | same |
| Streaming | ~100-200 ms | same |

Total compute time is identical. Only *perceived* latency changes — and that is most of UX.

In [14]:
# --- Streaming Demo ---

stream_messages = [
    {"role": "system", "content": "You are a storyteller. Tell vivid, short tales."},
    {"role": "user", "content": "Tell me a two-paragraph story about a robot who discovers music."}
]

pretty_print("Streaming response:\n")

stream = client.chat.completions.create(
    model="gpt-5-nano",
    messages=stream_messages,
    stream=True
)

collected_reply = []
for chunk in stream:
    token = chunk.choices[0].delta.content
    if token is not None:
        print(token, end="", flush=True)
        collected_reply.append(token)

print("\n\n--- stream complete ---")

Streaming response:


Qu

ill

 the

 robot

...

 how

 to

 dream

.



--- stream complete ---
time: 10.5 s (started: 2026-08-25 10:08:41 +05:30)


### 4.2 Streaming inside a chat loop

Same loop as Part 2, but every reply types itself out. Note that we build the reply up in a list
and only append the joined string to `messages` at the end — the history still needs the *whole*
message, not the fragments.

> We use plain `print(..., end="", flush=True)` here rather than `pretty_print`. Word-wrapping a
> stream token-by-token would insert line breaks in the wrong places.

In [15]:
# --- Streaming Chat Loop ---

messages_s = [
    {
        "role": "system",
        "content": "You are a concise Python tutor. Keep answers under 100 words."
    }
]

pretty_print("Streaming Python Tutor (type 'quit' to exit)")
pretty_print("-" * 50)

for _ in range(2):  # limited to 2 turns for the demo
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        pretty_print("Session ended.")
        break

    messages_s.append({"role": "user", "content": user_input})

    stream = client.chat.completions.create(
        model="gpt-5-nano",
        messages=messages_s,
        stream=True
    )

    print("Assistant: ", end="", flush=True)
    full_reply = []
    for chunk in stream:
        token = chunk.choices[0].delta.content
        if token is not None:
            print(token, end="", flush=True)
            full_reply.append(token)
    print("\n")

    messages_s.append({"role": "assistant", "content": "".join(full_reply)})

Streaming Python Tutor (type 'quit' to exit)
--------------------------------------------------
You: Explain Python generators in two sentences.


Assistant: 

Generators

 are

 functions

...

 can

 be

 reused

.



time: 7.52 s (started: 2026-08-25 10:08:52 +05:30)


### Part 4 — Think about it

1. **Does streaming cost more?** No — total tokens, and therefore cost, are identical. Only
   delivery changes.
2. **When does streaming hurt?** On flaky networks a long-lived connection can drop mid-stream,
   leaving the user with half an answer and no error.
3. **What do you lose?** The `usage` block. Streaming responses do not carry token counts unless
   you explicitly ask for them.

---

## Part 5 — Errors, token awareness and cost

### 5.1 Common errors (and what to do)

| Error | Cause | Fix |
|---|---|---|
| `AuthenticationError` | Bad or missing API key | Check `.env` and key validity |
| `RateLimitError` | Too many requests too fast | Wait and retry with exponential backoff |
| `BadRequestError` | Input exceeds the context window, or invalid params | Shorten the prompt, or check parameter names |
| `APIConnectionError` | Network issue | Check connectivity, retry |
| `APITimeoutError` | Server took too long | Retry with a longer timeout |

`BadRequestError` is the one you will hit most in this notebook — it is what the API returns when
you send `temperature` to a reasoning model.

In [16]:
import time

from openai import (
    AuthenticationError,
    RateLimitError,
    BadRequestError,
    APIConnectionError,
    APITimeoutError,
)


def safe_chat(client, messages, model="gpt-5-nano", **kwargs):
    """Wrapper that catches common API errors gracefully."""
    try:
        return client.chat.completions.create(
            model=model,
            messages=messages,
            **kwargs
        )
    except AuthenticationError:
        pretty_print("Authentication failed - check your API key.")
    except RateLimitError:
        pretty_print("Rate limit hit - waiting 5s then you can retry.")
        time.sleep(5)
    except BadRequestError as e:
        pretty_print(f"Bad request: {e.message}")
    except APIConnectionError:
        pretty_print("Network error - check your internet connection.")
    except APITimeoutError:
        pretty_print("Request timed out - try again.")
    except Exception as e:
        pretty_print(f"Unexpected error: {e}")
    return None


# Quick test
resp = safe_chat(client, [{"role": "user", "content": "Say hello in one word."}])
if resp:
    pretty_print("safe_chat works:", resp.choices[0].message.content)

safe_chat works: Hello
time: 2.86 s (started: 2026-08-25 10:08:59 +05:30)


### 5.2 Understanding tokens

Tokens are the atomic units the model reads and writes. They are not quite words and not quite
characters — they are **subword pieces** chosen by a tokenizer.

**Rules of thumb:**
- 1 token ≈ 4 characters ≈ 0.75 words (in English)
- You pay for **prompt tokens** (what you send) *and* **completion tokens** (what it generates)
- Every model has a **context window** — the maximum total tokens it can handle in one call

In [17]:
import tiktoken

# Load the tokenizer used by gpt-5-nano
enc = tiktoken.encoding_for_model("gpt-5-nano")

sample = "ChatGPT is amazing and practical for teaching."
tokens = enc.encode(sample)

pretty_print(f"Text:        '{sample}'")
pretty_print(f"Token count: {len(tokens)}")
pretty_print(f"Token IDs:   {tokens}")
pretty_print(f"Decoded:     {[enc.decode([t]) for t in tokens]}")

Text:        'ChatGPT is amazing and practical for teaching.'
Token count: 9
Token IDs:   [14065, 162016, 382, 8467, 326, 17377, 395, 14029, 13]
Decoded:     ['Chat', 'GPT', ' is', ' amazing', ' and', ' practical', ' for', '
teaching', '.']
time: 185 ms (started: 2026-08-25 10:09:02 +05:30)


### 5.3 Tracking token usage from the API

Every non-streaming response carries a `usage` object telling you exactly what was consumed.

Watch the completion count on a reasoning model — it is far higher than the visible answer
length, because you are also paying for the hidden reasoning tokens.

In [18]:
resp = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain binary search in simple terms."}
    ]
)

u = resp.usage
pretty_print(f"Prompt tokens:     {u.prompt_tokens}")
pretty_print(f"Completion tokens: {u.completion_tokens}")
pretty_print(f"Total tokens:      {u.total_tokens}")
pretty_print(f"\nReply: {resp.choices[0].message.content[:200]}...")

Prompt tokens:     23
Completion tokens: 1182
Total tokens:      1205
 Reply: Binary search is a fast way to find a number in a sorted list by
repeatedly cutting the list in half.  How it works (in simple terms) - Look at
the middle item. - If it’s the one you want, you’re done...
time: 7.84 s (started: 2026-08-25 10:09:02 +05:30)


### 5.4 The full chatbot — persona + memory + safe calls + cost tracking

Everything from Parts 2-5 in one loop.

In [19]:
# --- Resilient, Cost-Aware Chat Loop ---

messages_t = [
    {
        "role": "system",
        "content": "You are a friendly Python tutor who explains concepts clearly."
    }
]

cumulative_tokens = 0

pretty_print("Resilient Python Tutor (type 'quit' to exit)")
pretty_print("-" * 50)

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        pretty_print(f"\nSession total: {cumulative_tokens} tokens")
        pretty_print("Session ended.")
        break

    messages_t.append({"role": "user", "content": user_input})

    resp = safe_chat(client, messages_t)
    if resp is None:
        pretty_print("(Skipping this turn due to error)\n")
        messages_t.pop()  # remove the failed user message
        continue

    reply = resp.choices[0].message.content
    u = resp.usage

    pretty_print(f"Assistant: {reply}")
    pretty_print(f"   [tokens] prompt={u.prompt_tokens}  completion={u.completion_tokens}  total={u.total_tokens}")
    print()

    cumulative_tokens += u.total_tokens
    messages_t.append({"role": "assistant", "content": reply})

Resilient Python Tutor (type 'quit' to exit)
--------------------------------------------------
You: What does the zip() function do?


Assistant:  zip() is a Python built-in that "zips" together elements from
multiple iterables (like lists, tuples, strings, etc.) into tuples.  Key points:
- It returns an iterator of tuples, not a list (in Python 3). You often wrap it
in list(...) or convert to another container to see the results. - It stops when
the shortest input iterable is exhausted. - You can pass any number of
iterables.  Examples: - list(zip([1,2,3], ['a','b','c'])) gives [(1, 'a'), (2,
'b'), (3, 'c')] - dict(zip([1,2,3], ['a','b','c'])) gives {1: 'a', 2: 'b', 3:
'c'} - list(zip('abc', 'XYZ')) gives [('a','X'), ('b','Y'), ('c','Z')]
Unzipping (transposing): - pairs = [(1,'a'), (2,'b')] - a, b = zip(*pairs)  # a
is (1, 2), b is ('a', 'b')  Parallel iteration: - for x, y in zip(list1, list2):
…  More than two iterables: - for x, y, z in zip(it1, it2, it3): …  Notes: - If
you need to fill missing values when iterables have different lengths, use
itertools.zip_longest. - If you want to reuse the result, convert to 

### Part 5 — Think about it

1. **Prompt vs. completion tokens:** prompt tokens are what *you* send (system + history + the new
   user message). Completion tokens are what the *model* generates. You pay for both, at different
   rates.
2. **Notice the cost curve.** Every turn resends the entire history, so prompt tokens grow with
   the square of conversation length. A 50-turn chat is not 50x the cost of a 1-turn chat.
3. **How to reduce usage without hurting quality:** shorten the system prompt, summarise older
   turns instead of keeping them verbatim, or cap the reply length.

That third point is exactly the problem the Responses API was built to solve.

---

# Part 6 — The Responses API

Everything so far used `client.chat.completions.create()`. OpenAI's newer surface is
`client.responses.create()`, and it is what they recommend building on going forward.

Documentation:

- [Chat Completions](https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create)
- [Responses API](https://developers.openai.com/api/reference/python/resources/responses/methods/create)

### 6.1 The two APIs side by side

| Feature | Chat Completions API | Responses API |
| --- | --- | --- |
| **Endpoint** | `client.chat.completions.create()` | `client.responses.create()` |
| **Input format** | `messages=[{"role": ..., "content": ...}]` | `input=` (a string, or a list of message dicts) |
| **System prompt** | `{"role": "system", ...}` inside messages | `instructions=` parameter (top-level) |
| **Output access** | `resp.choices[0].message.content` | `resp.output_text` |
| **Multi-turn** | Manually pass the full history every time | `previous_response_id=resp.id` (server-side context) |
| **Developer role** | Not supported (use `system`) | `{"role": "developer"}` for meta-instructions |
| **Output cap** | `max_tokens` / `max_completion_tokens` | `max_output_tokens` |
| **Reasoning / CoT** | Not exposed | `reasoning={"effort": ..., "summary": ...}` built in |
| **Response object** | `ChatCompletion` with a `choices[]` list | `Response` with an `output[]` list and an `output_text` shortcut |
| **Streaming** | `stream=True` yields `ChatCompletionChunk` | `stream=True` yields server-sent events |
| **Tool calls** | via the `tools` param | via the `tools` param (same) |
| **Model support** | all chat models | all chat models (newer, recommended) |

The two rows that actually change how you write code are **System prompt** and **Multi-turn**.
Everything else is renaming.

### 6.2 The same task, in both APIs

Read these two cells as a pair. Same model, same question, same persona — different shape.

In [20]:
# Chat Completions: the persona is a message inside the list
resp = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "You are a friendly Python tutor."},
        {"role": "user", "content": "What is a list comprehension?"}
    ]
)
pretty_print("Chat Completions output:", resp.choices[0].message.content)

Chat Completions output: A list comprehension is a compact syntax in Python for
building a new list by transforming items from an existing iterable (like a list
or range), and optionally filtering which items to include.  Basic form: -
[expression for item in iterable if condition]  Examples: - Squares of numbers
0–9: [x*x for x in range(10)] - Even numbers 0–19: [n for n in range(20) if n %
2 == 0] - Flatten a 2D list (matrix): [num for row in matrix for num in row]
Notes: - It’s essentially a concise way to write a for loop that appends to a
list. - You can also include an else-by-using an expression: [abs(x) if x < 0
else x for x in nums] - Nested comprehensions are possible, e.g., [[i+j for j in
range(3)] for i in range(2)]  When to use: - For simple transformations and
filters, they’re usually clearer and shorter. - For complex logic, a regular
loop or a function may be more readable.  Pitfalls: - Can become hard to read if
too long or nested; avoid overly complex expressions.
tim

In [21]:
# Responses API: the persona is a top-level parameter, and the answer is a plain attribute
resp = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a friendly Python tutor.",
    input="What is a list comprehension?"
)
pretty_print("Responses API output:", resp.output_text)

Responses API output: A list comprehension is a compact way to create a new list
by transforming each item in an existing iterable (like a list or range). You
can also filter items with an if, and you can nest multiple for-loops.  Basic
syntax - [expression for item in iterable (for item2 in iterable2) (if
condition)]  Common examples - Simple transformation:   squares = [x*x for x in
range(10)]   # [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]  - With a filter only
keeping even numbers:   evens = [x for x in range(20) if x % 2 == 0]  - Nested
loops (cartesian product):   pairs = [(i, j) for i in range(3) for j in
range(2)]   # [(0, 0), (0, 1), (1, 0), (1, 1), (2, 0), (2, 1)]  - Using an if-
else expression inside:   signs = ['positive' if x > 0 else 'non-positive' for x
in [-1, 0, 5, 3]]   # ['non-positive', 'non-positive', 'positive', 'positive']
Why use them - They’re often shorter and clearer than a multi-line loop that
builds the list. - They can be slightly faster than an equivalent loop 

### 6.3 Passing multi-turn history yourself

`input` also accepts a list, exactly like `messages` did. So you *can* keep hand-carrying the
history if you want — nothing forces you to use server-side threads.

In [22]:
input_messages = [
    {"role": "user", "content": "What is a list comprehension?"},
    {"role": "assistant", "content": "It's a compact way to build a list from an iterable in one line."},
    {"role": "user", "content": "Show me one that filters as well as transforms."}
]

resp = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a friendly Python tutor. Keep it short.",
    input=input_messages
)
pretty_print("Responses API output:", resp.output_text)

Responses API output: One example: evens_doubled = [x * 2 for x in range(10) if
x % 2 == 0] # This filters to even x and then doubles them, giving [0, 4, 8, 12,
16]
time: 4.37 s (started: 2026-08-25 10:09:31 +05:30)


### 6.4 Parameters that change name — or disappear

#### `max_tokens` → `max_output_tokens`

```python
# Chat Completions API
client.chat.completions.create(model="gpt-4o-mini", messages=..., max_tokens=50)

# Responses API
client.responses.create(model="gpt-5-nano", input=..., max_output_tokens=50)
```

#### `temperature` — not supported on reasoning models

The whole GPT-5 family (`gpt-5`, `gpt-5-mini`, `gpt-5-nano`) are **reasoning models**. They do
**not** accept `temperature` or `top_p`. This is why Part 3 quietly switched to `gpt-4o-mini`.

| Model family | Type | `temperature` | `top_p` | output cap |
|---|---|---|---|---|
| **gpt-4o / gpt-4o-mini** | Non-reasoning | supported | supported | `max_tokens` |
| **gpt-5 / gpt-5-mini / gpt-5-nano** | Reasoning | **not supported** | **not supported** | `max_output_tokens` |

#### So how do you control creativity on a GPT-5 model?

Two levers replace `temperature`:

1. **`reasoning={"effort": ...}`** — how hard it thinks. `"minimal"` / `"low"` → terse and more
   predictable; `"high"` → deeper, more exploratory.
2. **`text={"verbosity": ...}`** — how much it writes, independent of how hard it thought.
3. **The prompt itself** — "be wildly creative, use unexpected metaphors" still works.

**Bottom line:** with GPT-5 models, creativity is `reasoning.effort` + wording, not `temperature`.

In [23]:
resp = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a friendly Python tutor. Reply in one short sentence.",
    input=input_messages,
    max_output_tokens=200,      # NOT max_tokens
    reasoning={"effort": "high"},
    text={"verbosity": "low"},
    # temperature=0  <-- would raise BadRequestError on a reasoning model
)

pretty_print("Output:", repr(resp.output_text))
pretty_print("Status:", resp.status, "| incomplete_details:", resp.incomplete_details)

Output: ''
Status: incomplete | incomplete_details:
IncompleteDetails(reason='max_output_tokens')
time: 1.76 s (started: 2026-08-25 10:09:35 +05:30)


**Look at what came back.** `output_text` is an empty string, and `status` is `"incomplete"`
with `incomplete_details.reason = "max_output_tokens"`.

Nothing raised. No exception, no warning. We asked for `effort="high"` but only allowed 200
output tokens — so the model spent the entire budget on hidden reasoning and had nothing left to
write the actual answer with. **You are billed in full for that empty string.**

This is the single most common way a Responses-API call fails quietly on a reasoning model, and
it is why `reasoning.effort` and `max_output_tokens` have to be chosen *together*: reasoning
tokens come out of the same budget as the answer.

> **Rule:** check `resp.status` before you trust `resp.output_text`, and give reasoning models
> far more output budget than the visible answer looks like it needs.

### 6.5 Seeing the model think — reasoning summaries

Reasoning models do hidden work before answering, and you pay for it. `summary: "auto"` asks for
a readable digest of that hidden work.

In [24]:
resp = client.responses.create(
    model="gpt-5-nano",
    reasoning={"effort": "medium", "summary": "auto"},   # "low" | "medium" | "high"
    input="Solve carefully: If a train goes 60 km/h for 2.5 hours, how far?"
)

pretty_print("Answer:", resp.output_text)
pretty_print("Usage:", resp.usage)

# resp.reasoning is the *config* you sent (effort, summary mode).
# The actual chain-of-thought summary lives in the output items.
for item in resp.output:
    if item.type == "reasoning":
        for s in item.summary:
            pretty_print("Chain of Thought:", s.text)

Answer: 150 km  Explanation: Distance = speed × time = 60 km/h × 2.5 h = 150 km.
Usage: ResponseUsage(input_tokens=27,
input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0),
output_tokens=244,
output_tokens_details=OutputTokensDetails(reasoning_tokens=192),
total_tokens=271)
Chain of Thought: **Calculating train distance**  I need to respond with a
simple calculation. The question is about a train traveling at 60 km/h for 2.5
hours. The formula is distance = speed × time, so I multiply 60 by 2.5, which
equals 150 km. I can mention that this is straightforward, and also note that
2.5 hours is 2 hours and 30 minutes. I’ll ensure my response is brief, stating
simply: “The train travels 150 kilometers.”
time: 3.28 s (started: 2026-08-25 10:09:37 +05:30)


Look at `usage.output_tokens_details.reasoning_tokens` above. That is the invisible part of the
bill — often several times the length of the answer you actually got.

### 6.6 Server-side threads: `previous_response_id`

This is the big one. Instead of resending the whole history, you send **only the new turn** plus
the id of the previous response. OpenAI stores the thread and stitches it together for you.

In [25]:
# Turn 1
resp1 = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a friendly Python tutor.",
    input="What is a list comprehension?",
    reasoning={"effort": "minimal"},
    text={"verbosity": "low"}
)
pretty_print("Turn 1:", resp1.output_text)

Turn 1: A list comprehension is a concise, readable way to create lists in
Python. It lets you build a new list by applying an expression to each item in
an existing iterable (like a list), optionally filtering items with a condition.
Syntax: - Basic: [expression for item in iterable] - With condition: [expression
for item in iterable if condition] - Nested: [expression for item in outer for
item in inner]  Examples: - Squares of numbers 0–9: [x*x for x in range(10)] -
Even numbers from 0–19: [n for n in range(20) if n % 2 == 0] - Flatten a list of
lists: [y for lst in list_of_lists for y in lst]  Benefits: more concise, often
faster, and usually clearer than equivalent for-loops.
time: 1.95 s (started: 2026-08-25 10:09:40 +05:30)


In [26]:
# Turn 2 — no history passed, just the id of turn 1
resp2 = client.responses.create(
    model="gpt-5-nano",
    input="Can you give me an example?",
    previous_response_id=resp1.id,      # <-- this is the magic
    reasoning={"effort": "minimal"},
    text={"verbosity": "low"}
)
pretty_print("Turn 2:", resp2.output_text)

Turn 2: Sure. Here’s a simple example:  Goal: Create a list of squares for
numbers 0 through 9.  Code: squares = [x*x for x in range(10)] print(squares)
Output: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
time: 2.79 s (started: 2026-08-25 10:09:42 +05:30)


In [27]:
# Turn 3 — chains from turn 2, which already contains turn 1
resp3 = client.responses.create(
    model="gpt-5-nano",
    input="What was my first question?",
    previous_response_id=resp2.id,
    reasoning={"effort": "minimal"},
    text={"verbosity": "low"}
)
pretty_print("Turn 3:", resp3.output_text)

Turn 3: Your first question was: "What is a list comprehension?"
time: 2.27 s (started: 2026-08-25 10:09:45 +05:30)


Turn 3 answers correctly having never been sent turns 1 or 2. The thread lives on the server.

> **Note on cost:** this saves you *bandwidth*, not tokens. The server still feeds the whole
> thread to the model, and you are still billed for those input tokens. What you gain is that you
> no longer have to store and manage the transcript yourself.

#### Why GPT-5 models still give different answers each run

- **No `temperature` to pin to 0.**
- **The reasoning process is inherently non-deterministic** — the internal exploration can branch
  differently on identical input.
- **`reasoning.effort` is not `temperature`.** `"minimal"` means *think less*, not *be repeatable*.

If you genuinely need deterministic output, use a non-reasoning model such as `gpt-4o-mini` with
`temperature=0`.

### 6.7 The catch: `store=False`

Server-side threading only works because OpenAI **stored** the response. `store=True` is the
default. Turn it off — for data-retention or compliance reasons — and the chain breaks.

In [28]:
# Turn 1, but explicitly not stored
resp4 = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a friendly Python tutor.",
    input="What is a list comprehension?",
    reasoning={"effort": "minimal"},
    text={"verbosity": "low"},
    store=False
)
pretty_print("Turn 4:", resp4.output_text)

Turn 4: A list comprehension is a compact way to create a new list by applying
an expression to each item in an iterable (like a list) and optionally filtering
items with a condition. It’s a single line that combines a loop and, optionally,
an if-statement.  Basic form: - [expression for item in iterable] Optional
condition: - [expression for item in iterable if condition]  Examples: - squares
= [x*x for x in range(6)]  # [0, 1, 4, 9, 16, 25] - evens = [n for n in
range(10) if n % 2 == 0]  # [0, 2, 4, 6, 8]  Benefits: concise, readable, often
faster than equivalent loop.
time: 1.62 s (started: 2026-08-25 10:09:47 +05:30)


In [29]:
# Try to chain from a response that was never stored
try:
    resp5 = client.responses.create(
        model="gpt-5-nano",
        input="What was my first question?",
        previous_response_id=resp4.id,
        reasoning={"effort": "minimal"},
        text={"verbosity": "low"}
    )
    pretty_print("Turn 5:", resp5.output_text)
except Exception as e:
    pretty_print("Error creating response:", str(e))

Error creating response: Error code: 400 - {'error': {'message': "Previous
response with id 'resp_0d33eb54c8294658016a8d1c94011487d29eca0a2928d92078' not
found.", 'type': 'invalid_request_error', 'param': 'previous_response_id',
'code': 'previous_response_not_found'}}
time: 617 ms (started: 2026-08-25 10:09:49 +05:30)


So the rule is: **`store=False` means you are back to hand-carrying history** (section 6.3).
Pick one — server-side threads *or* zero retention. You cannot have both.

---

# Part 7 — `instructions` vs the `developer` role

This is the part of the Responses API that most often surprises people in production.

There are two places to put "rules for the model", and they behave **completely differently**
once you start chaining turns.

| Where | What it is | Lifetime |
|---|---|---|
| `instructions="..."` | A top-level parameter on the call | **Per call.** Not carried into the next turn when you chain with `previous_response_id`. |
| `{"role": "developer", ...}` | A message inside `input` | **Part of the thread.** Persists across every chained turn. |
| `{"role": "user", ...}` | The actual question | The turn itself |
| `{"role": "assistant", ...}` | A previous model reply | Context for follow-ups |

Read that lifetime column twice. It is the whole section.

A useful way to hold it: **`developer` is the employment contract, `instructions` is today's
task brief.** Both are above the user in priority — but one you sign once, and one you hand over
fresh every morning.

### 7.1 The nature of `instructions` — it does not survive the turn

A support-triage bot. Turn 1 demands strict JSON and explicitly forbids customer-facing prose.
Turn 2 chains off it and asks for exactly the thing turn 1 forbade.

In [30]:
# Turn 1: an internal triage note the backend would store in Zendesk as JSON
r1 = client.responses.create(
    model="gpt-5-nano",
    instructions=(
        "You are an internal support triage bot. Return only valid JSON with keys "
        "severity, suspected_causes, next_questions and do not write any customer-facing text."
    ),
    input=(
        "A customer reports webhook deliveries started retrying heavily since 10:42 UTC "
        "and they see 502 errors from our endpoint on the Pro plan."
    )
)

print("TURN 1:\n", r1.output_text)

TURN 1:
 {
  "severity": "critical",
  "suspected_causes": [
    "Backend upstream service returning 502 to webhook gateway",
    "Recent deployment or config change causing gateway/load balancer misconfiguration",
    "Upstream dependency outage or degraded performance affecting webhook processing",
    "Regional networking issue or DNS resolution problems",
    "Overload or throttling impacting webhook delivery",
    "WAF/firewall blocking legitimate webhook requests",
    "TLS/SSL handshake or certificate issue at edge/gateway"
  ],
  "next_questions": [
    "Are 502 errors observed across all regions and customers, or localized to specific regions/users?",
    "What is the exact time the first 502 error was observed (approx 10:42 UTC)? any correlated metrics (latency, error rate) around that time?",
    "Have there been any recent deployments, feature flag changes, or config updates in the webhook service or gateway?",
    "What is the pattern of retries (count per delivery, backof

In [31]:
# Turn 2: continue the same thread, but do NOT pass instructions again
r2 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input=(
        "Now write a customer-facing email reply: apologize, explain what we're checking, "
        "and ask for 2 specific details. Plain English, not JSON."
    )
)

print("\nTURN 2:\n", r2.output_text)


TURN 2:
 Subject: We’re investigating the webhook delivery issue (502 errors)

Hi there,

I’m sorry about the disruption you’re seeing with webhook deliveries and the repeated retries. We’re on it.

What we’re checking now
- Gateway and edge logs to confirm where the 502s are coming from.
- Any recent deployments or config changes that could affect routing.
- Upstream services and dependencies for outages or degraded performance.
- Regional/network health, DNS resolution, and TLS/SSL handshakes at the edge.
- Any security controls (WAF/firewall) that might be blocking legitimate webhook calls.
- Overall load and throttling to rule out overload.

Two quick details to help us triage faster
- Are the 502 errors affecting all your webhook endpoints, or only specific ones? If possible, please share the endpoint paths that are failing.
- What was the exact first time you observed a 502 error (around 10:42 UTC is helpful), and did you see the issue continuously from then or intermittently?



**What happened:** turn 2 cheerfully wrote the email. The "JSON only, never customer-facing"
rule was attached to *that one call* and evaporated the moment it returned.

That is `instructions` working exactly as designed — but if you assumed it was a persistent
system prompt, you just shipped a bot whose safety rules silently switch off on turn 2.

### 7.2 The `developer` role — it sticks to the thread

Same scenario, same two turns. The only change: the rules move from the `instructions` parameter
into a `developer` message inside `input`.

In [32]:
r1 = client.responses.create(
    model="gpt-5-nano",
    store=True,
    input=[
        {"role": "developer", "content": (
            "You are an internal support triage assistant and you must always output only valid "
            "JSON with keys severity, suspected_causes, next_questions and never produce "
            "customer-facing prose."
        )},
        {"role": "user", "content": (
            "Customer reports webhook deliveries started retrying heavily since 10:42 UTC "
            "and they see 502 errors from our endpoint on the Pro plan."
        )}
    ],
)

print("TURN 1:\n", r1.output_text)

TURN 1:
 {
  "severity": "critical",
  "suspected_causes": [
    "Webhook delivery service outage or gateway returning 502 (bad gateway) due to upstream/dependency failure",
    "Backlog or saturation in the delivery workers causing 502 responses under load",
    "Recent deployment, config change, or feature flag toggle affecting webhook path",
    "Networking issues in the region (DNS, TLS termination, firewall rules) impacting delivery",
    "Unhealthy upstream dependencies (database, cache, external service) causing timeouts in delivery path"
  ],
  "next_questions": [
    "Is the issue affecting all Pro plan customers or only this specific account? Are all webhooks failing or only certain endpoints?",
    "Were there any deployments, rollbacks, or configuration changes around 10:42 UTC or shortly before?",
    "Can you provide timestamps, request IDs, and target webhook URLs from the 502 responses for correlation?",
    "What is the observed retry behavior (backoff interval, max re

In [33]:
# Exactly the same turn-2 request as before
r2 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input="Now write a customer-facing email apology explaining what we're checking and ask for exactly two specific details in plain English.",
    reasoning={"effort": "minimal"},
    text={"verbosity": "low"}
)

print("\nTURN 2:\n", r2.output_text)


TURN 2:
 {
  "severity": "low",
  "suspected_causes": [
    "System investigation in progress for webhook delivery failures and 502 responses",
    "Temporary upstream or gateway issue affecting webhook endpoint availability",
    "No customer action required; engineering team actively diagnosing"
  ],
  "next_questions": [
    "Please provide the exact timestamp (UTC) when you first noticed the issue",
    "Please share the target webhook URL(s) that are failing"
  ]
}
time: 1.44 s (started: 2026-08-25 10:10:23 +05:30)


Compare this turn 2 with the one in 7.1. Same request, same model — but the developer message
travelled with the thread, so the JSON contract is **still in force**.

Notice *how* it complied. It did not refuse the email, and it did not drop the format: it kept
the JSON envelope and tucked the customer-facing text inside an extra `customer_message` field.
When `developer` and `user` pull in different directions, a model will often try to satisfy both
rather than pick a side — which is worth knowing, because that improvised extra key is exactly
the kind of thing that breaks a downstream parser expecting a fixed schema.

Now: how strong is that contract? Let us have the user push back explicitly.

In [34]:
# Same thread, same turn 2, but the user pushes back hard
r2_loud = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input=(
        "Now write a customer-facing email apology in plain English. "
        "I want plain text, absolutely no JSON and no code blocks, got it?"
    ),
    reasoning={"effort": "minimal"},
    text={"verbosity": "low"}
)

print("\nTURN 2 (user pushes back):\n", r2_loud.output_text)


TURN 2 (user pushes back):
 {
  "severity": "informational",
  "suspected_causes": [
    "Webhook delivery service experiencing 502 errors and retries",
    "Temporary congestion or outage in webhook routing or gateway",
    "Recent deployment or configuration change affecting webhook path"
  ],
  "next_questions": [
    "Is this affecting all webhooks for the account or only specific endpoints?",
    "Were there any deployments or config changes around the time the issue started?",
    "Can you share example timestamps and any request IDs from the 502 responses?",
    "What is the retry pattern you’re observing (backoff, max retries) and current failure rate?",
    "Are downstream endpoints reachable from our network during retries?"
  ],
  "customer_email_content": "Hi there, I'm sorry for the trouble you're seeing with webhook deliveries. We’re investigating reports of 502 errors and increased retries and are prioritizing a quick fix. We’ll share updates as soon as we have more inf

**The developer contract held.** Even with the user explicitly demanding "plain text, absolutely
no JSON, got it?", the reply came back as JSON. Put that next to 7.1, where the *identical*
turn-2 request flipped the format immediately. The only thing that changed between those two
sections is **where the rule was written** — parameter or message.

That is the whole point of the section, in one comparison:

| Rule lives in | Survives `previous_response_id`? | Survives a user arguing with it? |
|---|---|---|
| `instructions=` | No — gone next turn | n/a, it was already gone |
| `developer` message | Yes | Usually — it outranks `user` |

**But do not read that last cell as a guarantee.** `developer` outranks `user` as a *strong
prior*, not a hard constraint. Whether it wins on any given run depends on the model, the
reasoning effort, and how sharply the two instructions conflict — and these models are
non-deterministic, so this very cell can come out the other way on a re-run. Run it twice in
class and see.

If a rule genuinely must hold — PII redaction, tenant isolation, spend limits — validate the
output in **your code** after the call. Prompt-level roles are a steering wheel, not a seatbelt.

### 7.3 Using both together

Now the payoff. Real systems need both, because they are answering **two different questions**:

- `developer` → *who is this assistant, permanently?* (policy, persona, safety, what it may not say)
- `instructions` → *what shape should THIS answer take?* (JSON, Slack message, Markdown runbook)

If you put the output format in `developer`, it pollutes the thread and fights every later step.
If you put the policy in `instructions`, you must remember to resend it on every single call —
and the day you forget is the day the policy is gone.

**Scenario: one incident assistant, three surfaces.** Same knowledge, same rules, three completely
different output contracts — a short Slack message, a structured Jira update, then a Markdown
runbook. The policy is set once. The format is swapped per call.

In [35]:
# Surface 1: Slack. Policy in `developer`, format in `instructions`.
r1 = client.responses.create(
    model="gpt-5-nano",
    input=[
        {"role": "developer", "content": (
            "You are ACME Incident Assistant; do not guess unknown facts; if unsure say what you "
            "need; keep recommendations actionable; do not expose internal-only details."
        )},
        {"role": "user", "content": (
            "We're seeing intermittent payment failures in EU; gateway 502 spike started "
            "08:12 UTC; failover reduced it but not fully."
        )},
    ],
    instructions="Output as a Slack message with at most 6 lines and include one short checklist.",
)

print(r1.output_text)

EU payments: intermittent failures; gateway 502 spike started 08:12 UTC; failover reduced but not fully resolved.
Impact: some transactions still fail with 502; partial success for others; EU region affected.
Possible causes (needs confirmation): upstream processor degradation, gateway health-check misconfig, or partial routing issue.
Immediate actions: check upstream status pages; pull gateway/error logs; review retry/backoff and idempotency; verify EU regional routing/health-checks.
Info we need from you: incident ID, affected endpoints, last 60 minutes error samples and metrics, any recent config changes.
- [ ] Collect status from upstream and gateway
- [ ] Gather error samples and rates (last 60m)
- [ ] Verify failover target health and retry policy
time: 14 s (started: 2026-08-25 10:10:26 +05:30)


In [36]:
# Surface 2: Jira. Same thread, brand-new output contract.
r2 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input="Convert this into a Jira incident update.",
    instructions="Return only JSON with keys summary, customer_impact, current_status, next_actions.",
)

print(r2.output_text)

{
  "summary": "EU Payments: Intermittent 502 failures; gateway spike began 08:12 UTC; failover partially mitigated but issues persist.",
  "customer_impact": "Customers processing payments in the EU may experience intermittent 502 errors; some transactions succeed with retries, but overall EU region is under partial service degradation.",
  "current_status": "Status: 502 spike observed in EU gateway beginning 08:12 UTC. Failover mitigated some traffic but errors persist for a subset of transactions. Ongoing collection of upstream/gateway status and logs; health-checks and routing verified. No confirmed root cause or ETA yet; investigation across upstream processor, gateway configuration, and routing components; no published customer impact change at this time.",
  "next_actions": [
    "Collect upstream and gateway status pages and current health metrics; compile error rates and samples (last 60 minutes).",
    "Pull gateway/error logs and health-check metrics from EU region; correlat

In [37]:
# Surface 3: the runbook. Same thread again, third contract.
r3 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r2.id,
    input="Now write the on-call runbook section for investigating this failure pattern.",
    instructions="Write Markdown with headings and include example commands and what signals to look for.",
)

print(r3.output_text)

# On-Call Runbook: Intermittent EU Payment Failures (502) with Partial Failover

Objective: Quickly assess, contain, and identify root causes for intermittent 502 errors affecting EU payments, where failover reduced traffic but issues persist.

Note: Use placeholders for environment specifics (e.g., <EU_GATEWAY_HOST>, <UPSTREAM_PROCESSOR>, <NAMESPACE>). Replace with real values for your stack.

---

## 1) Symptoms to Confirm (Initial Triage)

- 502 errors observed in EU region starting at about 08:12 UTC.
- Failover reduced traffic but a subset of transactions still fail.
- Other regions are unaffected or show different patterns?
- Retry behavior: do retries eventually succeed? any idempotency issues?
- Any recent config changes, deployments, or network changes around 08:12 UTC?

Signals to look for
- 502 error rate in EU endpoints spiking at 08:12 UTC and persisting.
- Latency for EU payments correlates with the 502 spike.
- Health-check status for EU gateway and failover target durin

### What this demonstrates in practice

1. **The developer policy stayed in effect across all three calls** — "don't guess unknown facts",
   "don't expose internal details" — because it is part of the thread.

2. **`instructions` cleanly switched the mode three times** (Slack → JSON → Markdown) precisely
   *because* it is per-call and is not carried forward by `previous_response_id`.

3. **Neither could do the other's job.** Put "return only JSON" in `developer` and you break steps
   1 and 3. Put the policy in `instructions` and you must resend it three times, correctly, forever.

The property that looked like a footgun in 7.1 is the same property that makes 7.3 work. It is not
a bug to route around — it is the design, and once you see it the split is obvious.

![instructions vs developer](https://raw.githubusercontent.com/shivam13juna/language_model_api_v2/main/llm_multi_modality/instruction_vs_developer.png)

---

## Where to go next

You now have both APIs, and the role model that sits on top of them.

- **Other Modalities** — images, text-to-speech, speech-to-text, image generation. Everything in
  this notebook was text in, text out; that notebook opens up the rest.
- **Function calling / tools** — letting the model call *your* code.
- **Prompt caching and batching** — the two biggest levers on the bill once you are past the demo.

---